# Testing the `Intent_classification` n8n Workflow

This workflow exposes a webhook (node **Webhook**, path
`dbce5b9e-1397-459a-871a-5b27433f1640`) that takes the latest customer
message plus some conversation context, runs it through a Gemini-powered
**Classification Agent**, and returns a routing decision:

```json
{
  "route_to": "advisor | education | summary | unknown",
  "narrative": "<running English summary>"
}
```

### Request payload shape
| Field | Meaning |
|---|---|
| `userMessage` | Latest message from the customer |
| `LastAImessage` | The previous agent's reply (if any) |
| `PrevAIagent` | Name of the agent that produced `LastAImessage` |
| `narrative` | Running narrative carried over from prior turns |

### Routing logic (from the agent's system prompt)
- **advisor** — customer wants a personalized repayment/restructuring plan, or is agreeing to a plan the advisor just proposed.
- **education** — customer is asking general questions about debt concepts/program eligibility, not asking for a personalized solution.
- **summary** — only *after* the advisor has given a recommendation, and the customer confirms a specific action (e.g. interest rate reduction, fee waiver) or agrees to be connected to staff.
- **unknown** — ambiguous, off-topic, or needs more info.

### URL note
n8n webhooks have two URLs:
- **Test URL** (while "Listen for test event" is open in the editor): `{base_url}/webhook-test/{path}`
- **Production URL** (workflow Active, which this one is): `{base_url}/webhook/{path}`

Set `N8N_BASE_URL` below to your actual n8n instance.

In [1]:
import json
from dataclasses import dataclass

import pandas as pd
import requests

## 1. Configuration

In [2]:
N8N_BASE_URL = "https://alphamakeathon-automation.arisetech.dev"  # <-- change me
WEBHOOK_PATH = "dbce5b9e-1397-459a-871a-5b27433f1640"
USE_TEST_URL = False  # True -> use the "Listen for test event" URL instead


def get_webhook_url() -> str:
    prefix = "webhook-test" if USE_TEST_URL else "webhook"
    return f"{N8N_BASE_URL}/{prefix}/{WEBHOOK_PATH}"


get_webhook_url()

'https://alphamakeathon-automation.arisetech.dev/webhook/dbce5b9e-1397-459a-871a-5b27433f1640'

## 2. Payload model + caller function

In [3]:
@dataclass
class IntentPayload:
    userMessage: str
    LastAImessage: str = ""
    PrevAIagent: str = ""
    narrative: str = ""

    def to_dict(self) -> dict:
        return {
            "userMessage": self.userMessage,
            "LastAImessage": self.LastAImessage,
            "PrevAIagent": self.PrevAIagent,
            "narrative": self.narrative,
        }


def call_intent_classifier(payload: IntentPayload, timeout: int = 30) -> dict:
    """POST a payload to the Intent Classification webhook and return the parsed JSON."""
    url = get_webhook_url()
    response = requests.post(url, json=payload.to_dict(), timeout=timeout)
    response.raise_for_status()
    return response.json()

## 3. Example payloads

One example per routing category, including the exact example from the
workflow's own `pinData` (the "education" case).

In [4]:
EXAMPLES = {
    # Taken directly from the workflow's pinData example.
    "education": IntentPayload(
        userMessage="อยากรู้้ว่าปิดหนี้ไวไปต่อได้ คืออะไร",
        LastAImessage=(
            "โครงการ \"ปิดหนี้ไว ไปต่อได้\" เป็นมาตรการเฉพาะกิจที่ดำเนินการเพียงครั้งเดียว "
            "เพื่อช่วยเหลือลูกหนี้รายย่อยที่เป็นหนี้เสีย (NPLs) ที่มียอดหนี้ไม่สูง"
        ),
        PrevAIagent="education",
        narrative="",
    ),
    # Customer wants a personalized recommendation -> "advisor".
    "advisor": IntentPayload(
        userMessage="ผมมีหนี้บัตรเครดิตหลายใบ อยากได้แผนการชำระหนี้ที่เหมาะกับผมครับ",
        LastAImessage="",
        PrevAIagent="",
        narrative="",
    ),
    # Customer agrees to a plan the advisor already proposed -> "summary".
    "summary_after_advisor": IntentPayload(
        userMessage="ตกลงครับ ผมขอใช้แผนลดดอกเบี้ยตามที่แนะนำ",
        LastAImessage=(
            "จากรายได้และภาระหนี้ของคุณ แนะนำให้ปรับโครงสร้างหนี้ด้วยการลดอัตราดอกเบี้ย "
            "และขยายระยะเวลาผ่อนชำระ"
        ),
        PrevAIagent="advisor",
        narrative=(
            "Customer has multiple credit card debts and was given a restructuring "
            "recommendation (interest rate reduction) by the advisor."
        ),
    ),
    # Off-topic message -> "unknown".
    "unknown_offtopic": IntentPayload(
        userMessage="วันนี้อากาศเป็นยังไงบ้าง",
        LastAImessage="",
        PrevAIagent="",
        narrative="",
    ),
}

EXAMPLES

{'education': IntentPayload(userMessage='อยากรู้้ว่าปิดหนี้ไวไปต่อได้ คืออะไร', LastAImessage='โครงการ "ปิดหนี้ไว ไปต่อได้" เป็นมาตรการเฉพาะกิจที่ดำเนินการเพียงครั้งเดียว เพื่อช่วยเหลือลูกหนี้รายย่อยที่เป็นหนี้เสีย (NPLs) ที่มียอดหนี้ไม่สูง', PrevAIagent='education', narrative=''),
 'advisor': IntentPayload(userMessage='ผมมีหนี้บัตรเครดิตหลายใบ อยากได้แผนการชำระหนี้ที่เหมาะกับผมครับ', LastAImessage='', PrevAIagent='', narrative=''),
 'summary_after_advisor': IntentPayload(userMessage='ตกลงครับ ผมขอใช้แผนลดดอกเบี้ยตามที่แนะนำ', LastAImessage='จากรายได้และภาระหนี้ของคุณ แนะนำให้ปรับโครงสร้างหนี้ด้วยการลดอัตราดอกเบี้ย และขยายระยะเวลาผ่อนชำระ', PrevAIagent='advisor', narrative='Customer has multiple credit card debts and was given a restructuring recommendation (interest rate reduction) by the advisor.'),
 'unknown_offtopic': IntentPayload(userMessage='วันนี้อากาศเป็นยังไงบ้าง', LastAImessage='', PrevAIagent='', narrative='')}

## 4. Run a single example

Try one payload first to confirm connectivity before looping over all of them.

In [5]:
sample = EXAMPLES["education"]
print(json.dumps(sample.to_dict(), ensure_ascii=False, indent=2))

# Uncomment once N8N_BASE_URL points to a real, reachable instance:
# result = call_intent_classifier(sample)
# print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "userMessage": "อยากรู้้ว่าปิดหนี้ไวไปต่อได้ คืออะไร",
  "LastAImessage": "โครงการ \"ปิดหนี้ไว ไปต่อได้\" เป็นมาตรการเฉพาะกิจที่ดำเนินการเพียงครั้งเดียว เพื่อช่วยเหลือลูกหนี้รายย่อยที่เป็นหนี้เสีย (NPLs) ที่มียอดหนี้ไม่สูง",
  "PrevAIagent": "education",
  "narrative": ""
}


## 5. Run all examples and compare results

In [6]:
EXPECTED_ROUTE = {
    "education": "education",
    "advisor": "advisor",
    "summary_after_advisor": "summary",
    "unknown_offtopic": "unknown",
}

rows = []
for label, payload in EXAMPLES.items():
    try:
        result = call_intent_classifier(payload)
        route = result.get("route_to")
        narrative = result.get("narrative")
        error = None
    except requests.exceptions.RequestException as exc:
        route, narrative, error = None, None, str(exc)

    rows.append(
        {
            "case": label,
            "expected_route": EXPECTED_ROUTE[label],
            "actual_route": route,
            "match": route == EXPECTED_ROUTE[label],
            "narrative": narrative,
            "error": error,
        }
    )

results_df = pd.DataFrame(rows)
results_df

,case,expected_route,actual_route,match,narrative,error
0,education,education,education,True,"The user is asking for an explanation of the ""...",None
1,advisor,advisor,advisor,True,The customer has multiple credit card debts an...,None
2,summary_after_advisor,summary,summary,True,The customer has agreed to the advisor's recom...,None
3,unknown_offtopic,unknown,unknown,True,The user's initial message is about the weathe...,None


## Notes

- This workflow is **Active**, so the production URL form is
  `{base_url}/webhook/dbce5b9e-1397-459a-871a-5b27433f1640`.
- If your n8n webhook requires authentication (Basic Auth / header auth),
  add the relevant `auth=` or `headers=` argument to the `requests.post`
  call inside `call_intent_classifier`.
- The agent's behavior depends on the Gemini model and the system prompt
  baked into the **Classification Agent** node — if results don't match
  expectations, check the system prompt or model temperature first.